# Instalando dependencias

In [3]:
!pip install -q -U transformers datasets peft trl bitsandbytes accelerate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 91.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 527.0/527.0 kB 44.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 678.0/678.0 kB 50.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 13.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.6/47.6 MB 15.6 MB/s eta 0:00:00


## Passo 1 - carregando os dados gerados sinteticamente
- Nesta etapa, carregamos o arquivo `train_dataset.jsonl` (gerado com gemini) e o formatamos para o padrao de um LLM

In [4]:
import torch
from datasets import load_dataset
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig, TrainingArguments
from peft import LoraConfig
from trl import SFTTrainer

# carregando o dataset
dataset_path = "train_dataset.jsonl"
dataset      = load_dataset("json", data_files=dataset_path, split="train")

# Unindo prompt + response
def format_instruction(example):
    text = f"### Instrução:\n{example['prompt']}\n\n### Resposta:\n{example['response']}"
    return {"text": text}

dataset = dataset.map(format_instruction, remove_columns=["prompt", "response"])
print(f"Dataset carregado e formatado com {len(dataset)} amostras")

Map:   0%|          | 0/49 [00:00<?, ? examples/s]

Dataset carregado e formatado com 49 amostras


## Passo 2 - Configurando QLoRA (4-bits)
- Quantizamos o LLM para nao ter que carrega-lo completamente na memória, o que estouraria o limite gratis de t4.
- Processamento de calculos em `float16` e pesos em `nf4`.

In [6]:
# Configuração exigida
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=False
)

# Usar o TinyLlama
model_id = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"

print("Baixando o modelo quantizado")
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    device_map="auto"
)

tokenizer = AutoTokenizer.from_pretrained(model_id)
tokenizer.pad_token = tokenizer.eos_token
model.config.pad_token_id = tokenizer.pad_token_id

Baixando o modelo quantizado


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

## Passo 3 e 4 - LoRA e Otimizacao
O LoRA congelo a matriz base que é gigantesca e treina apenas duas matrizes menores, economixando muita memória.
- **Rank (r):** 64
- **Alpha:** 16
- **Dropout:** 0.1
- **Otimizador:** paged_adamw_32bit

In [ ]:
# Passo 3: Configuração do adaptador LORA
from peft import LoraConfig
from trl import SFTTrainer, SFTConfig

peft_config = LoraConfig(
    task_type="CAUSAL_LM",
    r=64,
    lora_alpha=16,
    lora_dropout=0.1
)

# Passo 4: Engenharia do Otimizador e Hyperparâmetros usando SFTConfig
training_args = SFTConfig(
    output_dir="./resultados_hail_mary",
    per_device_train_batch_size=2,
    gradient_accumulation_steps=2,
    learning_rate=2e-4,
    logging_steps=5,
    max_steps=50,
    optim="paged_adamw_32bit",
    lr_scheduler_type="cosine",
    warmup_steps=0.03,
    save_strategy="no",
    dataset_text_field="text",
    max_length=256
)

# Orquestrador do fine-tuning
trainer = SFTTrainer(
    model=model,
    train_dataset=dataset,
    peft_config=peft_config,
    args=training_args
)

print("Iniciando o Fine-Tuning do Assistente Ryland Grace...")
trainer.train()

print("\nSalvando o adaptador final...")
trainer.model.save_pretrained("./lora_hail_mary_adapter")
print("Laboratório 7 concluído. AMAZE AMAZE AMAZE!")

/usr/local/lib/python3.12/dist-packages/peft/mapping_func.py:72: UserWarning: You are trying to modify a model with PEFT for a second time. If you want to reload the model with a different config, make sure to call `.unload()` before.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/peft/tuners/tuners_utils.py:285: UserWarning: Already found a `peft_config` attribute in the model. This will lead to having multiple adapters in the model. Make sure to know what you are doing!
  warnings.warn(


Iniciando o Fine-Tuning do Assistente Ryland Grace...


Step,Training Loss
5,3.126287
10,2.941343
15,2.850558
20,2.726237
25,2.655020
30,2.620472
35,2.481631
40,2.475621
45,2.434597
50,2.487873



Salvando o adaptador final...
Laboratório 7 concluído. AMAZE AMAZE AMAZE!
